In [1]:
%pip install --upgrade pip
%pip install  duckdb
%pip install   numpy
%pip install  pandas
%pip install fastavro
%pip install pyspark
%pip install pyarrow matplotlib

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 10.8 MB/s eta 0:00:00 0:00:01
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 8.5 MB/s eta 0:00:0000:010:01
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 MB 14.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 14.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 16.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 12.8 MB/s eta 0:00:00a 0:00:01
Note: 

In [2]:
#spark.stop()
from pyspark.sql import SparkSession
from pyspark import SparkConf, SparkContext
# from pyspark.sql.functions import col, count, avg, sum, min, max, desc
import sys
import time
import os
from pyspark.sql import SparkSession

# 3.5 means spark
# 2.12 means scala
# 1.10.1 - iceberg version
spark = SparkSession.builder \
    .appName("MinIO Write") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.spark:spark-avro_2.12:3.5.3,org.apache.spark:spark-core_2.12:3.5.3,org.apache.spark:spark-sql_2.12:3.5.3,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.10.1,org.apache.iceberg:iceberg-aws-bundle:1.10.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkSessionCatalog") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.rest_prod", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.rest_prod.type", "rest") \
    .config("spark.sql.catalog.rest_prod.uri", "http://rest:8181") \
    .config("spark.sql.catalog.rest_prod.warehouse", "s3://warehouse") \
    .config("spark.sql.defaultCatalog", "rest_prod") \
    .config("spark.sql.catalog.rest_prod.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.rest_prod.s3.path.style.access", "true") \
    .config("spark.sql.catalog.rest_prod.s3.connection.ssl.enabled", "false") \
    .config("spark.sql.catalog.rest_prod.s3.endpoint", "http://minio:9000") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.memoryOverhead", "2g") \
    .config("spark.driver.memoryOverhead", "1g") \
    .getOrCreate()


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.spark#spark-avro_2.12 added as a dependency
org.apache.spark#spark-core_2.12 added as a dependency
org.apache.spark#spark-sql_2.12 added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-abf70d2b-2518-4d8c-a7ab-ac99ce6bd8dd;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.apache.spark#spark-avro_2.12;3.5.3 in central
	found org.tukaani#xz;1.9 in central
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.1 in central
	found org.apache.iceberg

In [3]:
%%time

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/workspace/data.csv")

df.count()

CPU times: user 75 ms, sys: 15.9 ms, total: 90.9 ms
Wall time: 1min 9s


7160831

In [4]:
spark.sql("CREATE DATABASE IF NOT EXISTS rest_prod.warehouse.db")

DataFrame[]

In [5]:
%%time
df.writeTo("warehouse.data").createOrReplace()

26/03/13 13:59:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


CPU times: user 28.7 ms, sys: 3.65 ms, total: 32.3 ms
Wall time: 28.5 s


In [6]:
%%time
df2 = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", "\t") \
    .csv("/workspace/data2.tsv")

df2.count()

CPU times: user 30.2 ms, sys: 5.91 ms, total: 36.1 ms
Wall time: 12.9 s


5069140

In [16]:
from pyspark.sql.functions import monotonically_increasing_id, current_timestamp

In [17]:
%%time
df2.withColumn("id", monotonically_increasing_id()).writeTo("warehouse.data2").createOrReplace()

CPU times: user 41.4 ms, sys: 12.7 ms, total: 54.1 ms
Wall time: 26.6 s


In [18]:
%ls /workspace/minio-data/

ls: cannot access '/workspace/minio-data/': No such file or directory


In [19]:
from pathlib import Path

def get_dir_size_gb(path):
    """Return directory size in GB using pathlib"""
    return sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file()) / (1024**3)

# Usage
size_gb = get_dir_size_gb('/workspace/minioData/')
print(f"{size_gb:.2f} GB")

1.35 GB


In [20]:
%%time

df2 = spark.read.table("warehouse.data2")
df2.count()

CPU times: user 2.01 ms, sys: 2.26 ms, total: 4.27 ms
Wall time: 932 ms


5069140

In [21]:
%%time
from pyspark.sql.functions import col
df2.filter(col("id") == 51539620450).select("id", "customer_id", "product_title", "star_rating").count()

CPU times: user 4.5 ms, sys: 1.38 ms, total: 5.88 ms
Wall time: 925 ms


1

In [22]:
%%time
df2.filter(col("id").isin(51539620450, 223338334409, 231928311101)) \
  .select("id", "customer_id", "star_rating", "review_headline").count()

CPU times: user 5.3 ms, sys: 3.27 ms, total: 8.57 ms
Wall time: 755 ms


3

In [23]:
%%time
df2.filter((col("id") >= 50000000000) & (col("id") <= 100000000000)) \
  .select("id", "customer_id").count()

CPU times: user 3.16 ms, sys: 2.29 ms, total: 5.45 ms
Wall time: 1.04 s


1151928

In [28]:
from pyspark.sql.functions import col, count, avg, sum, min, max, desc

In [29]:
%%time
df2.filter((col("id") >= 50000000000) & (col("id") <= 100000000000)) \
  .select(sum(col("star_rating"))).show()

+----------------+
|sum(star_rating)|
+----------------+
|         4977220|
+----------------+

CPU times: user 4.06 ms, sys: 2.01 ms, total: 6.07 ms
Wall time: 871 ms


In [31]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", "\t") \
    .csv("/workspace/data1.tsv")

from pyspark.sql.functions import monotonically_increasing_id, current_timestamp


df.withColumn("id", monotonically_increasing_id() + 10000000).writeTo("warehouse.data1").createOrReplace()

In [32]:
%%time
df1 = spark.read.table("warehouse.data1")

df1.writeTo("warehouse.data2").append()

CPU times: user 31.7 ms, sys: 10.8 ms, total: 42.5 ms
Wall time: 19.6 s


26/03/13 14:06:19 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-d1919004-9dd7-4d36-8685-14c0251c0535/userFiles-6f291216-0a3c-4d51-94cb-8e530ac0a036. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-d1919004-9dd7-4d36-8685-14c0251c0535/userFiles-6f291216-0a3c-4d51-94cb-8e530ac0a036
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:174)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.SparkEnv.stop(SparkEnv.scala:108)
	at org.apache.spark.SparkContext.$anonfun$stop$25(SparkContext.scala:2305)
	at org.apac

In [ ]:
%%time
df1 = spark.read.table("warehouse.data1").filter((col("id") > 0) & (col("id") < 77309418381))

df1.writeTo("warehouse.data2").append()

In [ ]:
%%time
df1 = spark.read.table("warehouse.data1").filter((col("id") > 77309418381) & (col("id") < 154618836762))

df1.writeTo("warehouse.data2").append()

In [ ]:
%%time
df1 = spark.read.table("warehouse.data1").filter((col("id") > 154618836762))

df1.writeTo("warehouse.data2").append()